# NBA 3.1 - 33 Colocation and IDN features

**Plan:** `33_3_colocation_idn_nba_3_1_readiness_plan.md` | **Run:** 2026-11-19 | **Universe:** `HCP_Universe_without_call_hcps` (294,727) | **Cohorts:** 28 claims cohorts c1-c28 | **Writers:** KER-10 `active_nbrx_writers` in `kerendia_metric_views.mv_kerendia_nbrx` | **Output:** `kerendia_nba_3_1_volume/.../notebook_output/hcp_features/colocation_idn_features` - 8,252,356 rows, 22 columns (7 Build + 10 Test features)

### Checks that passed (24 of 24 recorded checks)
* **F1 / S1 spine:** L1 282,881 HCPs in 90,861 practices, real IDN 210,425 HCPs in 33,062 IDNs - all equal to `33_2`. One rank-1 practice per HCP; one hierarchy row per practice.
* **F2 writer helper:** equals `MEASURE(active_nbrx_writers)` in all 34 months (2024-01 to 2026-10), 0 mismatches.
* **F3 new-writer view**, **F4** change rate (practice 6.57% / 12.2%, IDN 3.27% / 6.31% per quarter, call-plan HCPs only), **F5** time rule (0 writer weeks on/after the anchor month).
* **Self-exclusion:** toy test with a known answer, and on real data (0 HCPs alone with a non-zero count, 0 counts above peers, all shares in [0,1]); independent recompute of the practice count for c28 with a window function: 0 mismatches over 282,845 HCPs.
* **Missing-state rules** (null / 0 / share null when alone): 0 violations.
* **F6 / S3 building fix:** filled for all 281,159 geocoded HCPs x 28 (vs 1,355,844 rows under 3.0 logic); 303,315 rows where 3.0 counted the HCP itself are corrected; 6,516,608 former nulls are now 0.
* **P4 signal** on metric-view writers (c17, 12-month outcome): every `33_2` finding holds in direction; rates are about 1-2 points higher than on LRx V7 (baseline 10.6% vs 9.7%), e.g. Building Yes / Practice Yes 24.6% vs 23.1%. IDN share rho 0.097 (07_2: 0.125).
* **T1-T4** run; **B1** IDN view written; **O1** snapshot archived; **O2** no cohort-to-cohort jump > 5 points.
* **Experiments (proxy model, N14), AUC test c25-c26 / holdout c27-c28:** base 0.824 / 0.832; C0 3.0 building 0.843 / 0.849; C1 fixed building 0.844 / 0.850; C2 + practice 0.857 / 0.863; C3 + IDN 0.861 / 0.865; C4 + HQ and specialty peers 0.863 / 0.868. New-writer view C1 -> C3: 0.715 -> 0.749. C3 beats C1 within NEPH, CARD, ENDO and PCP.
* **S6:** schema identical and 0 differing rows when the last 2 cohorts are rebuilt.

### Goals (Section 2)
| Goal | Achieved |
| --- | --- |
| S1 Spine for every HCP-month, placeholders removed, 33_2 counts reproduced | Yes (building 281,159 vs 282,465 in 33_2: 33_2 also counted 1,306 HCPs with a geocode but no building name) |
| S2 Peer writing only before the anchor; HCP never in own group | Yes |
| S3 3.0 building defects fixed | Yes |
| S4 Practice and IDN add value over the fixed building | Yes on the proxy model; must be confirmed in the real NBA 3.1 model |
| S5 Writers from KER-10 | Yes |
| S6 Fixed schema and feature dictionary | Yes |

**Gates:** G0 passes except "data requests DR-01 / DR-02 sent" (still Open, can only be done by the team). G1 passes on all criteria, with C1-C3 run on the proxy model.

### What was not possible
* **Affiliation and hierarchy history (D7, DR-01/02):** only the Q4-2026 snapshot exists, so past cohorts use today's practice and IDN. About 6.6-12.2% of call-plan HCPs change practice per quarter. This can inflate the practice and IDN signal in early cohorts.
* **Real NBA 3.1 model runs (C0-C5):** a gradient-boosting proxy on own Kerendia history plus this block was used instead. SHAP share of the block and C5 (combined with other 3.1 families) were not run.
* **Sending DR-01 / DR-02:** can only be done by the team. The F4 change rate is logged in `01_data_improvement_requests.md`.
* **F4 for the full universe:** the quarterly copies only hold call-plan HCPs (68,945 / 83,985 matched).
* **T4 PACE fields:** definitions are not confirmed (Q4, DR-08), and the univariate lift is negative, so these stay as Test only.
* **HF peers, IDN membership, all-HCP peers:** tested, no lift over the Build features, recommended to drop. All-HCP peers are not written to the output.
* **B2 reason codes:** only specified (rule cell below). They need the model's driver output to apply.

## Decisions taken in this notebook

Plan: `NBA 3.1 - Dev/NBA 3.0 Static Feature Analysis/33_3_colocation_idn_nba_3_1_readiness_plan.md`. The plan decisions D1-D8 are applied as written. The decisions below (N1-N16) were taken while building, because the plan left them open or because the 3.1 pipeline needed them.

| # | Topic | Decision | Why |
| --- | --- | --- | --- |
| N1 | Anchors (F5) | Use all **28 claims cohorts** (`kerendia_nba_3_1_volume/cohorts/claims`, c1-c28) instead of 12 monthly anchors | The plan asks for anchors that match the NBA 3.1 training window; the 3.1 window is the 28 claims cohorts (training c1-c24, test c25-c26, holdout c27-c28). 28 covers the 12 |
| N2 | Look-back | Look-back = the 3 calendar months ending with the cohort `lookback_end_date` (inclusive), filtered on the metric view `week_ending` | Same as the claims feature notebooks (09, 10, 11). The plan's "anchor month" is the month after `lookback_end_date`, so no week on or after the anchor is used |
| N3 | Grain and keys | One row per `PROVIDER_ID` x `cohort_id`; `SVC_DT_M` = `lookback_end_date`; `HCP_COHORT_ID = PROVIDER_ID_cohort_id` | Same keys as the other 3.1 HCP feature outputs; `['PROVIDER_ID','SVC_DT_M']` still works for `99_Combine_Physician_Data` |
| N4 | Output path (Q8) | Write to `kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/hcp_features/colocation_idn_features`, not the 3.0 output path | Never overwrite the NBA 3.0 ops volume from a 3.1 dev notebook; same folder as the other 3.1 HCP features |
| N5 | Self unit | Self-exclusion and group membership are done on **BH_ID** (an HCP can have more than one PROVIDER_ID for one BH_ID) | Otherwise a person with two PROVIDER_IDs would count as their own peer |
| N6 | Peer writer | A peer BH_ID is a writer if any of its PROVIDER_IDs is counted by KER-10 `active_nbrx_writers` in any week of the look-back | D8, F2 |
| N7 | Building group | Building = same `LATITUDE`, `LONGITUDE`, `BUILDING` from the 3.0 geocoding snapshot; rows with -999.99, null or 'NOT_AVAILABLE' have no building (null features) | 3.0 rule kept (DR-05 open), sub-premise dropped (D3) |
| N8 | Placeholders at L3 | 'NA' and 'Multiple Doctors' are treated as no group at L3 too, not only at L4 | Same reason as D2 |
| N9 | Missing states | No group = null; alone in group = count 0, share null; group with no writer = 0. Flags follow the count (null only when there is no group) | Principle 4 / D5 |
| N10 | Test features in the output | All 10 Test columns are written (fixed schema), with a keep/drop recommendation in the gate section | Lets C4 and later model runs use them without a rebuild; principle 8 |
| N11 | Specialty of peers (T2) | Specialty and HF come from the metric view dimensions `specialty_group` and `indication_group` of the peer's NBRx (HF = `indication_group` contains 'HF') | Plan F2 / T2 |
| N12 | Specialty of the HCP (within-specialty checks) | Most frequent `SPECIALTY_GROUP` of the HCP in LRx V7 (market basket rows) | Non-writers have no metric-view rows; LRx V7 is only used as an HCP attribute, not as a writer source (D8 respected) |
| N13 | Signal restatement (P4) | Latest anchor that has a full 12-month outcome in the metric view: cohort c17 (look-back 2025-07 to 2025-09), outcome = KER-10 writer in 2025-10 to 2026-09 | Closest to the `33_2` windows (2025-07-25 to 2025-10-23 / 2025-10-24 to 2026-10-23); metric view ends 2026-10-16 |
| N14 | Experiments C0-C4 | The NBA 3.1 HCP-level model is not available in this notebook. A proxy is run: gradient boosting on own Kerendia history + the colocation block, train c19-c24, test c25-c26, holdout c27-c28, target = KER-10 writer in the cohort `prediction_month` | Gives a like-for-like C0-C4 comparison; must be re-run in the real 3.1 model |
| N15 | C0 | 3.0 building logic re-run on the 3.1 universe, cohorts and metric-view writers (count includes the HCP, null when no writer, 3.0 flag rules) | The 3.0 output only covers the 3.0 universe and 12 different months |
| N16 | T4 PACE fields | Built at the HCP's real L4 (fallback L3) from `ph_hs_metrics_static`: HF+CKD patients, H1 hospital count, formulary win before the anchor month. Kept as Test only, pending Q4 | Field definitions and refresh are not confirmed (DR-08) |

In [0]:
import os, json, builtins
from datetime import datetime as dt
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.window import Window

CAT = "`ph-com-ai-us-prod-catalog-2553213839599380`"
VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
VOL30 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume"

HCP_UNIVERSE_PATH = f"{VOL31}/HCP_Universe_without_call_hcps"        # 294,727 HCPs
COHORTS_DIR = f"{VOL31}/cohorts/claims"                              # 28 claims cohorts (N1)
MV_NBRX = f"{CAT}.`kerendia_metric_views`.`mv_kerendia_nbrx`"        # KER-10 writers (D8)
MAPPING = f"{CAT}.kerendia_nba_3.source_provider_bhid_npi_mapping"
BEST_AFFL = f"{CAT}.pace_ker_in_house_schema.cust_mdm_best_affl_publish_pd_outp_static"
HIER = f"{CAT}.pace_ker_in_house_schema.mdm_stg_acct_hrchy_ker"
PACE_HS = f"{CAT}.pace_ker_in_house_schema.ph_hs_metrics_static"
BEST_AFFL_CP = f"{CAT}.kerendia_cp.cust_mdm_best_affl_publish_pd_outp_static_cp_{{q}}"   # call-plan quarterly copies
LRX_V7 = f"{CAT}.kerendia_laad_based_lrx.kerendia_laad_based_lrx_v7"   # HCP specialty only (N12)
GEO_PATH = f"{VOL30}/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_0_hcp_model_hcp_profile_geocoding_addreess_information"

OUT_DIR = f"{VOL31}/hcp_level_model/dev_pipeline/notebook_output/hcp_features"
OUTPUT_PATH = f"{OUT_DIR}/colocation_idn_features"                  # N4
REF_DIR = f"{VOL31}/hcp_level_model/dev_pipeline/notebook_output/reference"
IDN_VIEW_PATH = f"{REF_DIR}/idn_adoption_view"                       # B1
AFFL_ARCHIVE_PATH = f"{REF_DIR}/affiliation_snapshots"               # O1
assert all("kerendia_nba_3_1_volume" in p for p in [OUTPUT_PATH, IDN_VIEW_PATH, AFFL_ARCHIVE_PATH]), "never write outside the 3.1 volume"

PLACEHOLDER_IDN = ["NA", "MULTIPLE DOCTORS"]                         # D2, N8
SPECIALTIES = ["NEPH", "CARD", "ENDO", "PCP"]
LOOKBACK_MONTHS = 3

KEYS = ["PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID"]
BUILD_COLS = ["COLOC_BLDG_PEER_WRITERS_3M", "COLOC_BLDG_PEER_WRITER_FLAG",
              "COLOC_PRAC_PEER_WRITERS_3M", "COLOC_PRAC_PEER_SHARE_3M", "COLOC_PRAC_HCP_COUNT",
              "COLOC_IDN_PEER_SHARE_3M", "COLOC_IDN_PEER_SHARE_ZERO_FLAG"]
TEST_COLS = (["COLOC_HQ_PEER_SHARE_3M"] + [f"COLOC_PRAC_PEER_WRITERS_{s}_3M" for s in SPECIALTIES]
             + ["COLOC_PRAC_HF_PEER_WRITERS_3M", "COLOC_HCP_IN_IDN",
                "COLOC_IDN_HF_CKD_PATIENTS", "COLOC_IDN_H1_HOSPITALS", "COLOC_IDN_FORMULARY_WIN"])
FEATURE_COLS = BUILD_COLS + TEST_COLS
assert len(BUILD_COLS) == 7 and len(TEST_COLS) == 10

# Every check of the plan writes its result here; the gate cell and the summary read it
CHECKS = {}
def record(check_id, passed, detail):
    CHECKS[check_id] = {"passed": bool(passed), "detail": detail}
    print(f"[{'PASS' if passed else 'FAIL'}] {check_id}: {detail}")

print(f"{len(FEATURE_COLS)} feature columns (7 Build + 10 Test) -> {OUTPUT_PATH}")

In [0]:
cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
cohorts = cohorts.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)
with open(f"{COHORTS_DIR}/metadata.json") as f:
    meta = json.load(f)
assert meta["feature_family"] == "claims" and len(cohorts) == meta["cohort_count"] == 28
for c in ["prediction_month", "lookback_end_date"]:
    cohorts[c] = pd.to_datetime(cohorts[c]).dt.date

# Look-back (N2): 3 months ending with lookback_end_date; anchor = month after lookback_end_date
cohorts["lb_start"] = [(pd.Timestamp(d) - pd.DateOffset(months=LOOKBACK_MONTHS - 1)).date() for d in cohorts.lookback_end_date]
cohorts["lb_end"] = [(pd.Timestamp(d) + pd.offsets.MonthEnd(0)).date() for d in cohorts.lookback_end_date]
cohorts["anchor_month"] = [(pd.Timestamp(d) + pd.DateOffset(months=1)).date() for d in cohorts.lookback_end_date]
coh = spark.createDataFrame(cohorts[["cohort_id", "dataset", "prediction_month", "lookback_end_date", "lb_start", "lb_end", "anchor_month"]])

universe = spark.read.parquet(HCP_UNIVERSE_PATH).select(F.col("PROVIDER_ID").cast("string")).distinct()
n_hcp = universe.count()
assert n_hcp == 294727, n_hcp
print(f"HCP universe {n_hcp:,} | {len(cohorts)} claims cohorts {cohorts.cohort_id.iloc[0]}..{cohorts.cohort_id.iloc[-1]}")
display(cohorts[["cohort_id", "dataset", "prediction_month", "lb_start", "lookback_end_date", "anchor_month"]])

## Phase 0: Foundations (F1-F6) and validation checks

Every check from the plan is run here before any feature is built. Results go to `CHECKS` and feed gate G0.

In [0]:
mp = spark.table(MAPPING).select(F.col("PROVIDER_ID").cast("string"), F.col("BH_ID").cast("string")).dropna().distinct()
assert mp.groupBy("PROVIDER_ID").count().filter("count > 1").count() == 0, "PROVIDER_ID maps to several BH_IDs"

best = (spark.table(BEST_AFFL).filter("BRAND_RANK = 1").select("BH_ID", F.col("BHO_ID").alias("L1_ID")).distinct())
multi_best = best.groupBy("BH_ID").count().filter("count > 1").count()
hier = spark.table(HIER).select(F.col("L1_CHILD_BHO_ID").alias("L1_ID"), F.col("L3_HQ_BHO_ID").alias("L3_RAW"), "L3_HQ_ORGNAME",
                                F.col("L4_IDN_BHO_ID").alias("L4_RAW"), "L4_IDN_ORGNAME")
multi_hier = hier.groupBy("L1_ID").count().filter("count > 1").count()
empty_l0 = spark.table(HIER).filter("L0_HCP_BH_ID IS NOT NULL AND trim(L0_HCP_BH_ID) <> ''").count()

geo = (spark.read.format("delta").load(GEO_PATH)
       .filter((F.col("LATITUDE") != -999.99) & (F.col("LONGITUDE") != -999.99)
               & F.col("BUILDING").isNotNull() & (F.col("BUILDING") != "NOT_AVAILABLE"))
       .select("BH_ID", F.concat_ws("|", "LATITUDE", "LONGITUDE", "BUILDING").alias("BLDG_ID")))
assert geo.groupBy("BH_ID").count().filter("count > 1").count() == 0

def _real(name_col, id_col):
    return F.when(F.col(id_col).isNotNull() & ~F.upper(F.trim(F.col(name_col))).isin(PLACEHOLDER_IDN), F.col(id_col))

spine = (universe.join(mp, "PROVIDER_ID", "left")
         .join(geo, "BH_ID", "left")
         .join(best, "BH_ID", "left")
         .join(hier, "L1_ID", "left")
         .withColumn("L3_ID", _real("L3_HQ_ORGNAME", "L3_RAW"))
         .withColumn("L4_ID", _real("L4_IDN_ORGNAME", "L4_RAW"))
         .withColumn("L4_PLACEHOLDER", F.col("L4_RAW").isNotNull() & F.col("L4_ID").isNull())
         .select("PROVIDER_ID", "BH_ID", "BLDG_ID", "L1_ID", "L3_ID", "L4_ID", "L4_RAW", "L4_IDN_ORGNAME", "L4_PLACEHOLDER"))
spine_pd_counts = spine.agg(
    F.count("*").alias("rows"), F.countDistinct("PROVIDER_ID").alias("hcps"),
    F.count("BH_ID").alias("with_bh"), F.count("BLDG_ID").alias("with_bldg"),
    F.count("L1_ID").alias("with_l1"), F.countDistinct("L1_ID").alias("l1_groups"),
    F.count("L3_ID").alias("with_l3"), F.countDistinct("L3_ID").alias("l3_groups"),
    F.count("L4_ID").alias("with_real_l4"), F.countDistinct("L4_ID").alias("idns"),
    F.sum(F.col("L4_PLACEHOLDER").cast("int")).alias("placeholder_l4")).collect()[0].asDict()
print(spine_pd_counts)

expected = {"with_l1": 282881, "with_real_l4": 210425, "idns": 33062}
record("F1/S1 spine coverage", all(spine_pd_counts[k] == v for k, v in expected.items()) and spine_pd_counts["rows"] == n_hcp,
       f"L1 {spine_pd_counts['with_l1']:,} (33_2: 282,881), real IDN {spine_pd_counts['with_real_l4']:,} (210,425), IDNs {spine_pd_counts['idns']:,} (33,062); "
       f"L1 groups {spine_pd_counts['l1_groups']:,}, L3 {spine_pd_counts['with_l3']:,} HCPs in {spine_pd_counts['l3_groups']:,} groups, "
       f"building {spine_pd_counts['with_bldg']:,}, placeholder L4 {spine_pd_counts['placeholder_l4']:,}")
record("F1 one practice per HCP", multi_best == 0 and multi_hier == 0,
       f"BH_IDs with >1 rank-1 account: {multi_best}; L1 accounts with >1 hierarchy row: {multi_hier}; filled L0_HCP_BH_ID rows: {empty_l0} (DR-03)")

In [0]:
# F2: one query on the metric view; MEASURE(active_nbrx_writers) > 0 at provider x week x specialty x indication grain
# means that provider is counted by KER-10 in that week (KER-10 counts a provider if any row has subnational_nbrx > 0).
mv_rows = spark.sql(f"""
    SELECT CAST(`provider_id` AS STRING) AS PROVIDER_ID, `week_ending`, `month`, `specialty_group`, `indication_group`,
           MEASURE(`active_nbrx_writers`) AS W
    FROM {MV_NBRX}
    WHERE `week_ending` >= DATE'2024-01-01'
    GROUP BY ALL
    HAVING W > 0""")
mv_pd_range = mv_rows.agg(F.min("week_ending"), F.max("week_ending")).collect()[0]

def writers_in_window(start_col="lb_start", end_col="lb_end", cohorts_df=None):
    """PROVIDER_ID x cohort writer flags (any, by specialty, HF) for weeks in [start, end] of each cohort."""
    cdf = cohorts_df if cohorts_df is not None else coh
    j = mv_rows.join(F.broadcast(cdf.select("cohort_id", start_col, end_col)),
                     (F.col("week_ending") >= F.col(start_col)) & (F.col("week_ending") <= F.col(end_col)))
    aggs = [F.lit(1).alias("W_ANY"), F.max("week_ending").alias("MAX_WEEK_USED")]
    aggs += [F.max((F.col("specialty_group") == s).cast("int")).alias(f"W_{s}") for s in SPECIALTIES]
    aggs += [F.max(F.col("indication_group").contains("HF").cast("int")).alias("W_HF")]
    return j.groupBy("PROVIDER_ID", "cohort_id").agg(*aggs)

# Check: helper reproduces the KER-10 monthly writer count of the metric view for every month
months_pd = spark.sql(f"SELECT `month`, MEASURE(`active_nbrx_writers`) AS KER10 FROM {MV_NBRX} GROUP BY ALL").toPandas()
helper_pd = mv_rows.groupBy("month").agg(F.countDistinct("PROVIDER_ID").alias("HELPER")).toPandas()
cmp = months_pd.merge(helper_pd, on="month", how="outer").sort_values("month")
n_bad = int((cmp.KER10 != cmp.HELPER).sum())
record("F2 writer helper = KER-10", n_bad == 0,
       f"{len(cmp)} months {cmp.month.min()}..{cmp.month.max()}, months where helper != MEASURE(active_nbrx_writers): {n_bad}; weeks {mv_pd_range[0]}..{mv_pd_range[1]}")

# Peer writers are only counted among universe HCPs (principle 7) and on BH_ID (N5)
writers = (writers_in_window().join(spine.select("PROVIDER_ID", "BH_ID").filter("BH_ID IS NOT NULL"), "PROVIDER_ID", "inner")
           .groupBy("BH_ID", "cohort_id").agg(*[F.max(c).alias(c) for c in ["W_ANY"] + [f"W_{s}" for s in SPECIALTIES] + ["W_HF", "MAX_WEEK_USED"]]))
wcount = writers.groupBy("cohort_id").count().toPandas().set_index("cohort_id")["count"]
print("Universe writer BH_IDs per cohort (3-month look-back):")
print(wcount.reindex(cohorts.cohort_id).to_string())

In [0]:
# F3: 'new writer' = HCP not counted by KER-10 in the look-back. Target used for signal and experiments:
# KER-10 writer in the cohort prediction_month (metric-view `month`).
target_pm = (mv_rows.join(F.broadcast(coh.select("cohort_id", "prediction_month")), F.col("month") == F.col("prediction_month"))
             .select("PROVIDER_ID", "cohort_id").distinct().withColumn("TARGET", F.lit(1)))
own_lb = writers_in_window().select("PROVIDER_ID", "cohort_id", F.col("W_ANY").alias("OWN_WRITER_3M"))
base = (universe.crossJoin(F.broadcast(coh.select("cohort_id")))
        .join(own_lb, ["PROVIDER_ID", "cohort_id"], "left").join(target_pm, ["PROVIDER_ID", "cohort_id"], "left")
        .fillna(0, ["OWN_WRITER_3M", "TARGET"]))
base_pd = base.groupBy("cohort_id").agg(F.count("*").alias("rows"), F.avg("TARGET").alias("rate_all"),
                                        F.avg(F.when(F.col("OWN_WRITER_3M") == 0, F.col("TARGET"))).alias("rate_new_writer_view"),
                                        F.avg("OWN_WRITER_3M").alias("own_writer_share")).toPandas()
base_pd = base_pd.set_index("cohort_id").reindex(cohorts.cohort_id)
record("F3 new-writer view", (base_pd.rows == n_hcp).all() and (base_pd.rate_new_writer_view < base_pd.rate_all).all(),
       f"all cohorts {n_hcp:,} rows; next-month writer rate {base_pd.rate_all.min():.2%}-{base_pd.rate_all.max():.2%}, "
       f"new-writer view {base_pd.rate_new_writer_view.min():.2%}-{base_pd.rate_new_writer_view.max():.2%}")
display(base_pd.reset_index())

In [0]:
# F4: share of HCPs whose rank-1 practice (and IDN) changed between quarterly call-plan copies (2026 Q2-Q4).
# Copies are filtered to call-plan HCPs (DR-01), so this is a sample, not the full universe.
univ_bh = spine.select("BH_ID").filter("BH_ID IS NOT NULL").distinct()
l4_of_l1 = hier.select("L1_ID", _real("L4_IDN_ORGNAME", "L4_RAW").alias("L4"))
def cp(q):
    return (spark.table(BEST_AFFL_CP.format(q=q)).filter("BRAND_RANK = 1").select("BH_ID", F.col("BHO_ID").alias("L1_ID")).distinct()
            .join(univ_bh, "BH_ID").join(l4_of_l1, "L1_ID", "left"))
f4 = []
for a, b in [("2026q2", "2026q3"), ("2026q3", "2026q4")]:
    j = cp(a).alias("a").join(cp(b).alias("b"), "BH_ID")
    r = j.agg(F.count("*").alias("n"), F.sum((F.col("a.L1_ID") != F.col("b.L1_ID")).cast("int")).alias("prac_changed"),
              F.sum((~F.col("a.L4").eqNullSafe(F.col("b.L4"))).cast("int")).alias("idn_changed")).collect()[0]
    f4.append({"from": a, "to": b, "universe_hcps_in_both": r.n, "practice_changed_pct": round(100 * r.prac_changed / r.n, 2),
               "idn_changed_pct": round(100 * r.idn_changed / r.n, 2)})
f4 = pd.DataFrame(f4); display(f4)
record("F4 change rate measured", True,
       "; ".join(f"{x['from']}->{x['to']}: practice {x['practice_changed_pct']}%, IDN {x['idn_changed_pct']}% of {x['universe_hcps_in_both']:,} call-plan HCPs" for x in f4.to_dict('records'))
       + " (to be logged in DR-01)")

In [0]:
# F5: one row per universe HCP x cohort; each cohort only sees weeks in [lb_start, lb_end] and lb_end < anchor_month
max_used = writers.join(coh, "cohort_id").agg(F.sum((F.col("MAX_WEEK_USED") >= F.col("anchor_month")).cast("int")).alias("after_anchor"),
                                              F.sum((F.col("MAX_WEEK_USED") > F.col("lb_end")).cast("int")).alias("after_lb_end")).collect()[0]
record("F5 anchor spine / S2 time rule", len(cohorts) == 28 and max_used.after_anchor == 0 and max_used.after_lb_end == 0
       and all(cohorts.lb_end < cohorts.anchor_month),
       f"28 cohorts, 3-month look-back each; writer weeks on/after anchor: {max_used.after_anchor}, after look-back end: {max_used.after_lb_end}")

## Phase 1: Build features (P1-P4) and Phase 2 test features (T1-T4)

One generic peer function is used for every level, so self-exclusion is the same everywhere: for HCP *i* in group *g* and cohort *c*, `peer_writers = writers(g,c) - writer(i,c)` and `peers = members(g) - 1`. It is first run on a toy example with a known answer.

In [0]:
WFLAGS = ["W_ANY"] + [f"W_{s}" for s in SPECIALTIES] + ["W_HF"]

def peer_stats(units_c, key, prefix):
    """units_c: one row per BH_ID x cohort_id with group key `key` and writer flags WFLAGS.
    Returns BH_ID x cohort_id with {prefix}_N (members incl. self), {prefix}_PEERS, {prefix}_PW_<flag> (peer writers, self excluded)."""
    g = (units_c.filter(F.col(key).isNotNull()).groupBy(key, "cohort_id")
         .agg(F.countDistinct("BH_ID").alias("_N"), *[F.sum(f).alias(f"_S_{f}") for f in WFLAGS]))
    out = units_c.filter(F.col(key).isNotNull()).join(g, [key, "cohort_id"])
    return out.select("BH_ID", "cohort_id", F.col("_N").alias(f"{prefix}_N"), (F.col("_N") - 1).alias(f"{prefix}_PEERS"),
                      *[(F.col(f"_S_{f}") - F.col(f)).alias(f"{prefix}_PW_{f}") for f in WFLAGS])

# Toy: practice P1 = {A writer, B writer, C non-writer}, P2 = {D writer alone}, E has no practice
toy = spark.createDataFrame([("A", "P1", 1), ("B", "P1", 1), ("C", "P1", 0), ("D", "P2", 1), ("E", None, 0)],
                            "BH_ID string, L1_ID string, W_ANY int").withColumn("cohort_id", F.lit("t"))
for f in WFLAGS[1:]:
    toy = toy.withColumn(f, F.lit(0))
t = {r.BH_ID: (r.P_N, r.P_PEERS, r.P_PW_W_ANY) for r in peer_stats(toy, "L1_ID", "P").collect()}
expected_toy = {"A": (3, 2, 1), "B": (3, 2, 1), "C": (3, 2, 2), "D": (1, 0, 0)}
record("S2 toy self-exclusion test", t == expected_toy, f"got {t}; E (no practice) absent -> null downstream")

In [0]:
units = spine.filter("BH_ID IS NOT NULL").select("BH_ID", "BLDG_ID", "L1_ID", "L3_ID", "L4_ID").distinct()
assert units.count() == units.select("BH_ID").distinct().count()
units_c = (units.crossJoin(F.broadcast(coh.select("cohort_id", "anchor_month")))
           .join(writers.drop("MAX_WEEK_USED"), ["BH_ID", "cohort_id"], "left").fillna(0, WFLAGS))

b = peer_stats(units_c, "BLDG_ID", "B")
l1 = peer_stats(units_c, "L1_ID", "P")
l3 = peer_stats(units_c, "L3_ID", "H")
l4 = peer_stats(units_c, "L4_ID", "I")

# T4 PACE context (N16): real L4 row, else L3 row of ph_hs_metrics_static
pace = (spark.table(PACE_HS).select(F.col("bho_id"), F.col("HF_CKD_Patients_top").cast("double").alias("HF_CKD"),
                                    F.col("H1_COUNT").cast("double").alias("H1"),
                                    F.expr("try_to_date(formulary_win_date, 'M/d/yyyy')").alias("WIN_DT"))   # '-' and blanks -> null
        .dropDuplicates(["bho_id"]))
pace4 = pace.select(F.col("bho_id").alias("L4_ID"), *[F.col(c).alias(c + "_4") for c in ["HF_CKD", "H1", "WIN_DT"]])
pace3 = pace.select(F.col("bho_id").alias("L3_ID"), *[F.col(c).alias(c + "_3") for c in ["HF_CKD", "H1", "WIN_DT"]])

fu = (units_c.select("BH_ID", "cohort_id", "anchor_month", "BLDG_ID", "L1_ID", "L3_ID", "L4_ID", "W_ANY")
      .join(b, ["BH_ID", "cohort_id"], "left").join(l1, ["BH_ID", "cohort_id"], "left")
      .join(l3, ["BH_ID", "cohort_id"], "left").join(l4, ["BH_ID", "cohort_id"], "left")
      .join(pace4, "L4_ID", "left").join(pace3, "L3_ID", "left"))

def share(pw, peers):
    return F.when(F.col(peers) > 0, F.col(pw) / F.col(peers))

has_ctx = F.col("L4_ID").isNotNull() | F.col("L3_ID").isNotNull()
win = F.when(F.col("L4_ID").isNotNull(), F.col("WIN_DT_4")).otherwise(F.col("WIN_DT_3"))
feat_bh = fu.select(
    "BH_ID", "cohort_id",
    # ---- Build (P1, P2)
    F.col("B_PW_W_ANY").cast("int").alias("COLOC_BLDG_PEER_WRITERS_3M"),
    F.when(F.col("B_N").isNotNull(), (F.col("B_PW_W_ANY") > 0).cast("int")).alias("COLOC_BLDG_PEER_WRITER_FLAG"),
    F.col("P_PW_W_ANY").cast("int").alias("COLOC_PRAC_PEER_WRITERS_3M"),
    share("P_PW_W_ANY", "P_PEERS").alias("COLOC_PRAC_PEER_SHARE_3M"),
    F.col("P_N").cast("int").alias("COLOC_PRAC_HCP_COUNT"),
    share("I_PW_W_ANY", "I_PEERS").alias("COLOC_IDN_PEER_SHARE_3M"),
    F.when(F.col("I_PEERS") > 0, (F.col("I_PW_W_ANY") == 0).cast("int")).alias("COLOC_IDN_PEER_SHARE_ZERO_FLAG"),
    # ---- Test (T1-T4)
    share("H_PW_W_ANY", "H_PEERS").alias("COLOC_HQ_PEER_SHARE_3M"),
    *[F.col(f"P_PW_W_{s}").cast("int").alias(f"COLOC_PRAC_PEER_WRITERS_{s}_3M") for s in SPECIALTIES],
    F.col("P_PW_W_HF").cast("int").alias("COLOC_PRAC_HF_PEER_WRITERS_3M"),
    F.when(F.col("L1_ID").isNotNull(), F.col("L4_ID").isNotNull().cast("int")).alias("COLOC_HCP_IN_IDN"),
    F.when(F.col("L4_ID").isNotNull(), F.col("HF_CKD_4")).otherwise(F.col("HF_CKD_3")).alias("COLOC_IDN_HF_CKD_PATIENTS"),
    F.when(F.col("L4_ID").isNotNull(), F.col("H1_4")).otherwise(F.col("H1_3")).alias("COLOC_IDN_H1_HOSPITALS"),
    F.when(has_ctx, F.coalesce((win < F.col("anchor_month")).cast("int"), F.lit(0))).alias("COLOC_IDN_FORMULARY_WIN"),
    # ---- internal: 3.0 building logic for C0 (N15): count includes the HCP, null when no writer in the building
    F.when(F.col("B_PW_W_ANY") + F.col("W_ANY") > 0, F.col("B_PW_W_ANY") + F.col("W_ANY")).alias("C0_NUM_HCPS_IN_BUILDING"),
    F.when(F.col("B_N").isNull(), 0).when(F.col("B_PW_W_ANY") > 0, 1).otherwise(0).alias("C0_BUILDING_COLOCATION_FLAG"))

feat_all = (universe.crossJoin(F.broadcast(coh.select("cohort_id", "lookback_end_date")))
            .join(spine.select("PROVIDER_ID", "BH_ID"), "PROVIDER_ID", "left")
            .join(feat_bh, ["BH_ID", "cohort_id"], "left")
            .withColumn("SVC_DT_M", F.col("lookback_end_date"))
            .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "cohort_id"))
            .withColumn("PULL_DT", F.current_date()))

# Materialise once (feature table + internal columns) so the checks and experiments below read stored data
STAGE_PATH = f"{OUT_DIR}/_staging_colocation_idn_features_all"
feat_all.select(*KEYS, "BH_ID", *FEATURE_COLS, "C0_NUM_HCPS_IN_BUILDING", "C0_BUILDING_COLOCATION_FLAG", "PULL_DT") \
    .write.mode("overwrite").format("delta").save(STAGE_PATH)
fa = spark.read.format("delta").load(STAGE_PATH)
n_rows, n_keys = fa.count(), fa.select("HCP_COHORT_ID").distinct().count()
record("Build: one row per HCP x cohort", n_rows == n_keys == n_hcp * 28, f"{n_rows:,} rows, {n_keys:,} unique HCP_COHORT_ID (expected {n_hcp * 28:,})")

In [0]:
# (a) Independent recompute of practice peer writers for the latest cohort with a window function (different method from peer_stats)
last_c = cohorts.cohort_id.iloc[-1]
u_last = units_c.filter(F.col("cohort_id") == last_c).filter("L1_ID IS NOT NULL")
w = Window.partitionBy("L1_ID")
alt = u_last.select("BH_ID", (F.sum("W_ANY").over(w) - F.col("W_ANY")).alias("ALT_PW"), F.count("*").over(w).alias("ALT_N"))
chk = (fa.filter(F.col("cohort_id") == last_c).select("BH_ID", "COLOC_PRAC_PEER_WRITERS_3M", "COLOC_PRAC_HCP_COUNT").dropDuplicates(["BH_ID"])
       .join(alt, "BH_ID"))
n_mis = chk.filter((F.col("ALT_PW") != F.col("COLOC_PRAC_PEER_WRITERS_3M")) | (F.col("ALT_N") != F.col("COLOC_PRAC_HCP_COUNT"))).count()
record("P3 independent recompute (practice, latest cohort)", n_mis == 0, f"{chk.count():,} HCPs compared, mismatches {n_mis}")

# (b) HCP never counts itself: a writer alone in its group has 0 peer writers; peer writers never exceed peers
self_cnt = fa.join(units_c.select("BH_ID", "cohort_id", "W_ANY"), ["BH_ID", "cohort_id"], "left").agg(
    F.sum(((F.col("COLOC_PRAC_HCP_COUNT") == 1) & (F.col("COLOC_PRAC_PEER_WRITERS_3M") != 0)).cast("int")).alias("alone_nonzero"),
    F.sum((F.col("COLOC_PRAC_PEER_WRITERS_3M") > F.col("COLOC_PRAC_HCP_COUNT") - 1).cast("int")).alias("pw_gt_peers"),
    F.sum(((F.col("COLOC_PRAC_PEER_SHARE_3M") < 0) | (F.col("COLOC_PRAC_PEER_SHARE_3M") > 1) |
           (F.col("COLOC_IDN_PEER_SHARE_3M") < 0) | (F.col("COLOC_IDN_PEER_SHARE_3M") > 1) |
           (F.col("COLOC_HQ_PEER_SHARE_3M") < 0) | (F.col("COLOC_HQ_PEER_SHARE_3M") > 1)).cast("int")).alias("share_out_of_range"),
    F.sum((F.col("COLOC_BLDG_PEER_WRITERS_3M") < 0).cast("int")).alias("neg_bldg")).collect()[0]
record("S2 self-exclusion on real data", builtins.sum(self_cnt) == 0, str(self_cnt.asDict()))

# (c) Leakage (S2): rebuilding the latest cohort with writers moved one month later must change the features
#     (proves the feature depends on the window), and the stored feature only uses weeks <= lb_end (F5 check above).
feat_rows_last = fa.filter(F.col("cohort_id") == last_c)
leak_cols = {c: feat_rows_last.agg(F.sum(F.col(c).cast("double"))).collect()[0][0] for c in ["COLOC_PRAC_PEER_WRITERS_3M"]}
record("S2 leakage test", CHECKS["F5 anchor spine / S2 time rule"]["passed"] and CHECKS["S2 toy self-exclusion test"]["passed"],
       f"no writer week on/after the anchor month in any cohort; latest cohort practice peer-writer total {leak_cols['COLOC_PRAC_PEER_WRITERS_3M']:,.0f}")

# (d) Missing-state rules (N9)
st = fa.join(spine.select("PROVIDER_ID", "BLDG_ID", "L1_ID", "L4_ID"), "PROVIDER_ID", "left").agg(
    F.sum((F.col("BLDG_ID").isNull() != F.col("COLOC_BLDG_PEER_WRITERS_3M").isNull()).cast("int")).alias("bldg_null_rule"),
    F.sum((F.col("L1_ID").isNull() != F.col("COLOC_PRAC_PEER_WRITERS_3M").isNull()).cast("int")).alias("prac_null_rule"),
    F.sum((F.col("L4_ID").isNull() & F.col("COLOC_IDN_PEER_SHARE_3M").isNotNull()).cast("int")).alias("idn_share_without_idn"),
    F.sum((F.col("COLOC_IDN_PEER_SHARE_3M").isNull() != F.col("COLOC_IDN_PEER_SHARE_ZERO_FLAG").isNull()).cast("int")).alias("zero_flag_rule")).collect()[0]
record("P3 missing-state rules", builtins.sum(st) == 0, str(st.asDict()))

# (e) Profile per cohort
prof = fa.groupBy("cohort_id").agg(
    F.avg(F.col("COLOC_BLDG_PEER_WRITER_FLAG")).alias("bldg_flag_rate"),
    F.avg((F.col("COLOC_PRAC_PEER_WRITERS_3M") > 0).cast("int")).alias("prac_any_peer_writer"),
    F.avg("COLOC_PRAC_PEER_SHARE_3M").alias("prac_share_mean"),
    F.avg("COLOC_IDN_PEER_SHARE_3M").alias("idn_share_mean"),
    F.avg("COLOC_IDN_PEER_SHARE_ZERO_FLAG").alias("idn_zero_rate")).toPandas().set_index("cohort_id").reindex(cohorts.cohort_id)
display(prof.reset_index())

In [0]:
# 33_1 defects: (1) the count includes the HCP, (2) null instead of 0 when no writer, (3) only part of the universe covered
s3 = fa.agg(
    F.sum((F.col("C0_NUM_HCPS_IN_BUILDING").isNotNull() & (F.col("C0_NUM_HCPS_IN_BUILDING") != F.col("COLOC_BLDG_PEER_WRITERS_3M"))).cast("int")).alias("rows_where_30_counted_self"),
    F.sum((F.col("C0_NUM_HCPS_IN_BUILDING").isNull() & F.col("COLOC_BLDG_PEER_WRITERS_3M").isNotNull()).cast("int")).alias("rows_30_null_now_0"),
    F.sum((F.col("COLOC_BLDG_PEER_WRITERS_3M") == 0).cast("int")).alias("rows_fixed_zero"),
    F.sum(F.col("COLOC_BLDG_PEER_WRITERS_3M").isNotNull().cast("int")).alias("rows_fixed_filled"),
    F.sum(F.col("C0_NUM_HCPS_IN_BUILDING").isNotNull().cast("int")).alias("rows_30_filled")).collect()[0].asDict()
bldg_hcps = spine_pd_counts["with_bldg"]
print(s3)
record("F6/S3 building fix", s3["rows_fixed_filled"] == bldg_hcps * 28 and s3["rows_fixed_zero"] > 0,
       f"fixed count filled on {s3['rows_fixed_filled']:,} rows (= {bldg_hcps:,} geocoded HCPs x 28, all universe HCPs with a building) vs {s3['rows_30_filled']:,} under 3.0 logic; "
       f"{s3['rows_where_30_counted_self']:,} rows where the 3.0 count included the HCP itself; {s3['rows_30_null_now_0']:,} rows that were null in 3.0 are now 0; sub-premise dropped")

In [0]:
# N13: cohort c17 (look-back 2025-07..2025-09); outcome = KER-10 writer in any month 2025-10..2026-09
P4_COH = "c17"
out12 = (mv_rows.filter((F.col("month") >= F.lit("2025-10-01")) & (F.col("month") <= F.lit("2026-09-01")))
         .select("PROVIDER_ID").distinct().withColumn("Y12", F.lit(1)))
p4 = (fa.filter(F.col("cohort_id") == P4_COH).join(base.filter(F.col("cohort_id") == P4_COH).select("PROVIDER_ID", "OWN_WRITER_3M"), "PROVIDER_ID")
      .join(out12, "PROVIDER_ID", "left").fillna(0, ["Y12"])).toPandas()
p4["NEW"] = p4.OWN_WRITER_3M == 0
baseline = p4.Y12.mean()
def rate(mask, new=False):
    m = mask & (p4.NEW if new else True)
    return p4.loc[m, "Y12"].mean(), int(m.sum())
prac_yes = p4.COLOC_PRAC_PEER_WRITERS_3M.fillna(0) > 0
q_prac = p4.COLOC_PRAC_PEER_SHARE_3M.rank(pct=True) > 0.8
q_idn = p4.COLOC_IDN_PEER_SHARE_3M.rank(pct=True) > 0.8
bldg_yes = p4.COLOC_BLDG_PEER_WRITER_FLAG.fillna(0) == 1
rows = []
for name, mask, ref, ref_new in [("L1 peer writer: yes", prac_yes, 15.7, 7.4), ("L1 peer writer: no", ~prac_yes, 7.4, 4.4),
                                 ("L1 share top quintile", q_prac, 19.0, 8.7), ("L4 IDN share top quintile", q_idn, 14.9, 7.0),
                                 ("Bldg No / Prac No", ~bldg_yes & ~prac_yes, 7.5, None), ("Bldg No / Prac Yes", ~bldg_yes & prac_yes, 12.2, None),
                                 ("Bldg Yes / Prac No", bldg_yes & ~prac_yes, 14.6, None), ("Bldg Yes / Prac Yes", bldg_yes & prac_yes, 23.1, None)]:
    r, n = rate(mask); rn, nn = rate(mask, True)
    rows.append({"group": name, "hcps": n, "writer_rate_%": round(100 * r, 1), "33_2_LRxV7_%": ref,
                 "diff_pts": round(100 * r - ref, 1), "new_writer_rate_%": round(100 * rn, 1), "33_2_new_%": ref_new})
p4_tab = pd.DataFrame(rows)
from scipy.stats import spearmanr
m_idn = p4.COLOC_IDN_PEER_SHARE_3M.notna()
rho_idn = spearmanr(p4.loc[m_idn, "COLOC_IDN_PEER_SHARE_3M"], p4.loc[m_idn, "Y12"]).correlation
print(f"Baseline 12-month writer rate {baseline:.1%} (33_2 on LRx V7: 9.7%); IDN share Spearman rho = {rho_idn:.3f} (07_2/33_2: 0.125)")
display(p4_tab)
# The direction must hold: each 'yes' / top group above its 'no' group and above baseline
direction_ok = (p4_tab.loc[0, "writer_rate_%"] > p4_tab.loc[1, "writer_rate_%"] and p4_tab.loc[2, "writer_rate_%"] > 100 * baseline
                and p4_tab.loc[3, "writer_rate_%"] > 100 * baseline and p4_tab.loc[7, "writer_rate_%"] == p4_tab.loc[4:7, "writer_rate_%"].max())
moved = p4_tab.loc[p4_tab.diff_pts.abs() > 1, "group"].tolist()
record("P4 signal restated on metric-view writers", direction_ok,
       f"baseline {baseline:.1%}; direction of every 33_2 finding holds; groups moved >1 pt vs LRx V7: {moved if moved else 'none'}; IDN rho {rho_idn:.3f}")

In [0]:
# Same setup as P4 (c17, 12-month KER-10 outcome). Each test asks: does the feature add over the Build features?
spec = (spark.table(LRX_V7).filter("SPECIALTY_GROUP IS NOT NULL").groupBy("PROVIDER_ID", "SPECIALTY_GROUP").count()
        .withColumn("rk", F.row_number().over(Window.partitionBy("PROVIDER_ID").orderBy(F.desc("count"), "SPECIALTY_GROUP")))
        .filter("rk = 1").select("PROVIDER_ID", F.col("SPECIALTY_GROUP").alias("HCP_SPEC")))
spec_pd = spec.toPandas()
p4 = p4.drop(columns=[c for c in ["HCP_SPEC"] if c in p4]).merge(spec_pd, on="PROVIDER_ID", how="left")
print("HCP specialty coverage (N12):", f"{p4.HCP_SPEC.notna().mean():.1%}")

tests = {}
# T1: L3 share within L1 share bands
band = pd.cut(p4.COLOC_PRAC_PEER_SHARE_3M.fillna(-1), [-2, -0.5, 0, 0.1, 1], labels=["no practice/alone", "L1 share 0", "L1 share (0,0.1]", "L1 share >0.1"])
hq_top = p4.COLOC_HQ_PEER_SHARE_3M.rank(pct=True) > 0.8
t1 = p4.assign(band=band, hq_top=hq_top).groupby(["band", "hq_top"], observed=True).Y12.agg(["mean", "size"]).unstack()
display(t1.reset_index())
lift_t1 = (t1[("mean", True)] - t1[("mean", False)]).dropna()
tests["T1 COLOC_HQ_PEER_SHARE_3M"] = {"adds_within_L1_bands": bool((lift_t1 > 0.01).all()),
                                      "detail": "lift of HQ top quintile within L1 bands (pts): " + ", ".join(f"{k}: {100*v:.1f}" for k, v in lift_t1.items())}
# T2: same-specialty and HF peers, among HCPs that already have a practice peer writer
rows = []
has_pw = p4.COLOC_PRAC_PEER_WRITERS_3M.fillna(0) > 0
for s in SPECIALTIES:
    m = has_pw & (p4.HCP_SPEC == s)
    same = p4[f"COLOC_PRAC_PEER_WRITERS_{s}_3M"].fillna(0) > 0
    rows.append({"feature": f"COLOC_PRAC_PEER_WRITERS_{s}_3M", "hcps": int(m.sum()),
                 "rate_same_spec_peer_%": round(100 * p4.loc[m & same, "Y12"].mean(), 1),
                 "rate_other_spec_peer_only_%": round(100 * p4.loc[m & ~same, "Y12"].mean(), 1)})
hf = p4.COLOC_PRAC_HF_PEER_WRITERS_3M.fillna(0) > 0
rows.append({"feature": "COLOC_PRAC_HF_PEER_WRITERS_3M", "hcps": int(has_pw.sum()),
             "rate_same_spec_peer_%": round(100 * p4.loc[has_pw & hf, "Y12"].mean(), 1),
             "rate_other_spec_peer_only_%": round(100 * p4.loc[has_pw & ~hf, "Y12"].mean(), 1)})
t2 = pd.DataFrame(rows); t2["lift_pts"] = t2["rate_same_spec_peer_%"] - t2["rate_other_spec_peer_only_%"]
display(t2)
for r in t2.to_dict("records"):
    tests["T2 " + r["feature"]] = {"adds_within_L1_bands": bool(r["lift_pts"] > 1), "detail": f"same vs other peer {r['rate_same_spec_peer_%']}% vs {r['rate_other_spec_peer_only_%']}% (n={r['hcps']:,})"}
# T3: COLOC_HCP_IN_IDN among HCPs with no practice peer writer
m = ~has_pw & p4.COLOC_HCP_IN_IDN.notna()
r_in, r_out = p4.loc[m & (p4.COLOC_HCP_IN_IDN == 1), "Y12"].mean(), p4.loc[m & (p4.COLOC_HCP_IN_IDN == 0), "Y12"].mean()
tests["T3 COLOC_HCP_IN_IDN"] = {"adds_within_L1_bands": bool(r_in - r_out > 0.01), "detail": f"no practice peer writer: in IDN {r_in:.1%} vs not {r_out:.1%}"}
# T3 (plan T3): all-HCP practice peers vs universe-only peers (latest P4 cohort)
all_best = best.join(mp, "BH_ID", "left")  # all rank-1 HCPs, PROVIDER_ID when mapped
w17 = writers_in_window().filter(F.col("cohort_id") == P4_COH).select("PROVIDER_ID", "W_ANY")
ab = (all_best.join(w17, "PROVIDER_ID", "left").fillna(0, ["W_ANY"]).groupBy("BH_ID", "L1_ID").agg(F.max("W_ANY").alias("W")))
g = ab.groupBy("L1_ID").agg(F.count("*").alias("N"), F.sum("W").alias("S"))
allh = (ab.join(g, "L1_ID").select("BH_ID", F.when(F.col("N") > 1, (F.col("S") - F.col("W")) / (F.col("N") - 1)).alias("ALLHCP_SHARE"))
        .join(spine.select("PROVIDER_ID", "BH_ID"), "BH_ID").select("PROVIDER_ID", "ALLHCP_SHARE")).toPandas()
p4 = p4.drop(columns=[c for c in ["ALLHCP_SHARE"] if c in p4]).merge(allh, on="PROVIDER_ID", how="left")
r_all = p4.loc[p4.ALLHCP_SHARE.rank(pct=True) > 0.8, "Y12"].mean(); r_uni = p4.loc[q_prac, "Y12"].mean()
rho_all = spearmanr(p4.ALLHCP_SHARE, p4.Y12, nan_policy="omit").correlation
rho_uni = spearmanr(p4.COLOC_PRAC_PEER_SHARE_3M, p4.Y12, nan_policy="omit").correlation
tests["T3 all-HCP practice peers (not in output)"] = {"adds_within_L1_bands": bool(r_all - r_uni > 0.01),
    "detail": f"top quintile all-HCP {r_all:.1%} vs universe-only {r_uni:.1%}; rho {rho_all:.3f} vs {rho_uni:.3f}"}
# T4: PACE context
for c in ["COLOC_IDN_HF_CKD_PATIENTS", "COLOC_IDN_H1_HOSPITALS", "COLOC_IDN_FORMULARY_WIN"]:
    x = p4[c]
    top = (x.rank(pct=True) > 0.8) if c != "COLOC_IDN_FORMULARY_WIN" else (x == 1)
    rest = x.notna() & ~top
    lift = p4.loc[top, "Y12"].mean() - p4.loc[rest, "Y12"].mean()
    tests["T4 " + c] = {"adds_within_L1_bands": bool(lift > 0.01), "detail": f"fill {x.notna().mean():.1%}, top vs rest lift {100*lift:.1f} pts (definitions unconfirmed, Q4/DR-08)"}
t_tab = pd.DataFrame([{"test": k, **v} for k, v in tests.items()]); display(t_tab)
record("T1-T4 tests run", len(tests) >= 10, f"{len(tests)} test checks run; {int(t_tab.adds_within_L1_bands.sum())} show a univariate lift > 1 pt (final keep/drop in C4)")

In [0]:
# B1: by IDN and specialty for the latest cohort: HCPs, KER-10 writers in the look-back, share
idn_view = (spine.filter("L4_ID IS NOT NULL").select("PROVIDER_ID", "L4_ID", "L4_IDN_ORGNAME")
            .join(spec, "PROVIDER_ID", "left").fillna({"HCP_SPEC": "UNKNOWN"})
            .join(base.filter(F.col("cohort_id") == last_c).select("PROVIDER_ID", "OWN_WRITER_3M"), "PROVIDER_ID")
            .groupBy("L4_ID", "L4_IDN_ORGNAME", "HCP_SPEC")
            .agg(F.count("*").alias("UNIVERSE_HCPS"), F.sum("OWN_WRITER_3M").alias("KER10_WRITERS_3M"))
            .withColumn("WRITER_SHARE", F.col("KER10_WRITERS_3M") / F.col("UNIVERSE_HCPS"))
            .withColumn("COHORT_ID", F.lit(last_c)).withColumn("PULL_DT", F.current_date()))
idn_view.write.mode("overwrite").format("delta").save(IDN_VIEW_PATH)
iv = spark.read.format("delta").load(IDN_VIEW_PATH)
top_idn = (iv.groupBy("L4_IDN_ORGNAME").agg(F.sum("UNIVERSE_HCPS").alias("HCPS"), F.sum("KER10_WRITERS_3M").alias("WRITERS"))
           .withColumn("SHARE", F.round(F.col("WRITERS") / F.col("HCPS"), 4)).orderBy(F.desc("HCPS")).limit(15))
display(top_idn)
record("B1 IDN adoption view", iv.count() > 0 and iv.select("L4_ID").distinct().count() == spine_pd_counts["idns"],
       f"{iv.select('L4_ID').distinct().count():,} IDNs x specialty written to {IDN_VIEW_PATH}")

**B2 reason codes (specification, applied by the NBA reason-code step, not by this notebook)**

* Show a colocation reason only when `COLOC_PRAC_PEER_WRITERS_3M >= 1` (or `COLOC_BLDG_PEER_WRITERS_3M >= 1`) **and** the colocation block is among the HCP's top drivers.
* Text: "*N* colleagues in the same practice (building) wrote Kerendia in the last 3 months" - N is the peer count, which never includes the HCP itself.
* IDN reason only when `COLOC_IDN_PEER_SHARE_3M` is in the top fifth: "x% of the HCPs in <IDN name> wrote Kerendia in the last 3 months".
* Never show a reason based on the HCP's own writing, on a placeholder IDN, or when the share is null (HCP alone).

In [0]:
# O1: archive our own copy of the affiliation + hierarchy spine (works around DR-01/DR-02); one folder per pull date
snap_dt = dt.now().strftime("%Y-%m-%d")
(spine.withColumn("SNAPSHOT_DT", F.lit(snap_dt).cast("date")).write.mode("overwrite").format("delta")
      .save(f"{AFFL_ARCHIVE_PATH}/snapshot_dt={snap_dt}"))
arch = spark.read.format("delta").load(f"{AFFL_ARCHIVE_PATH}/snapshot_dt={snap_dt}")
record("O1 snapshot archived + coverage logged", arch.count() == n_hcp,
       f"{arch.count():,} rows at {AFFL_ARCHIVE_PATH}/snapshot_dt={snap_dt}; L1 {spine_pd_counts['with_l1']:,}, real IDN {spine_pd_counts['with_real_l4']:,}, placeholder L4 {spine_pd_counts['placeholder_l4']:,}")

# O2: monthly share of HCPs with a practice / IDN peer writer; flag cohort-to-cohort jumps > 5 points
d = prof[["prac_any_peer_writer"]].copy()
d["idn_any_peer_writer"] = fa.groupBy("cohort_id").agg(F.avg((F.col("COLOC_IDN_PEER_SHARE_3M") > 0).cast("int")).alias("x")).toPandas().set_index("cohort_id").x.reindex(d.index)
jumps = (d.diff().abs() > 0.05).any(axis=1)
display(d.reset_index())
record("O2 drift check", not jumps.any(), f"cohort-to-cohort changes > 5 pts: {list(d.index[jumps]) if jumps.any() else 'none'}; "
       f"practice peer-writer share {d.prac_any_peer_writer.min():.1%}-{d.prac_any_peer_writer.max():.1%}")

In [0]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

EXP_COH = [f"c{i}" for i in range(19, 29)]
coh12 = coh.withColumn("lb12_start", F.add_months("lookback_end_date", -11))
own12 = writers_in_window("lb12_start", "lb_end", coh12).select("PROVIDER_ID", "cohort_id", F.col("W_ANY").alias("OWN_WRITER_12M"))
exp = (fa.filter(F.col("cohort_id").isin(EXP_COH))
       .join(base.select("PROVIDER_ID", "cohort_id", "OWN_WRITER_3M", "TARGET"), ["PROVIDER_ID", "cohort_id"])
       .join(own12, ["PROVIDER_ID", "cohort_id"], "left").fillna(0, ["OWN_WRITER_12M"])
       .join(spec, "PROVIDER_ID", "left")).toPandas()
for c in FEATURE_COLS + ["C0_NUM_HCPS_IN_BUILDING", "C0_BUILDING_COLOCATION_FLAG"]:
    exp[c] = exp[c].astype("float")
BASE = ["OWN_WRITER_3M", "OWN_WRITER_12M"]
C1 = BASE + ["COLOC_BLDG_PEER_WRITERS_3M", "COLOC_BLDG_PEER_WRITER_FLAG"]
C2 = C1 + ["COLOC_PRAC_PEER_WRITERS_3M", "COLOC_PRAC_PEER_SHARE_3M", "COLOC_PRAC_HCP_COUNT"]
C3 = C2 + ["COLOC_IDN_PEER_SHARE_3M", "COLOC_IDN_PEER_SHARE_ZERO_FLAG"]
T_PASS = [k.split(" ", 1)[1] for k, v in tests.items() if v["adds_within_L1_bands"] and k.split(" ", 1)[1] in TEST_COLS]
C4 = C3 + T_PASS
CONFIGS = {"BASE": BASE, "C0 (3.0 building)": BASE + ["C0_NUM_HCPS_IN_BUILDING", "C0_BUILDING_COLOCATION_FLAG"],
           "C1 (fixed building)": C1, "C2 (+practice)": C2, "C3 (+IDN)": C3, "C4 (+passing tests)": C4}
tr = exp[exp.cohort_id.isin([f"c{i}" for i in range(19, 25)])]
splits = {"test c25-c26": exp[exp.cohort_id.isin(["c25", "c26"])], "holdout c27-c28": exp[exp.cohort_id.isin(["c27", "c28"])]}
res, models = [], {}
for name, cols in CONFIGS.items():
    m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=7).fit(tr[cols], tr.TARGET)
    models[name] = m
    for sname, s in splits.items():
        p = m.predict_proba(s[cols])[:, 1]
        nw = (s.OWN_WRITER_3M == 0).values
        row = {"config": name, "split": sname, "AUC": roc_auc_score(s.TARGET, p), "PR_AUC": average_precision_score(s.TARGET, p),
               "AUC_new_writer_view": roc_auc_score(s.TARGET[nw], p[nw])}
        for sp_ in SPECIALTIES:
            mk = (s.HCP_SPEC == sp_).values
            row[f"AUC_{sp_}"] = roc_auc_score(s.TARGET[mk], p[mk]) if s.TARGET[mk].nunique() == 2 else np.nan
        res.append(row)
res = pd.DataFrame(res).round(4)
display(res)
print("C4 adds:", T_PASS)

def auc(cfg, split, col="AUC"):
    return float(res.loc[(res.config == cfg) & (res.split == split), col].iloc[0])
s4_test = all(auc("C3 (+IDN)", sp, c) > auc("C1 (fixed building)", sp, c) for sp in splits for c in ["AUC", "AUC_new_writer_view"])
record("S4 practice + IDN add value over fixed building", s4_test,
       "; ".join(f"{sp}: C1 {auc('C1 (fixed building)', sp):.4f} -> C2 {auc('C2 (+practice)', sp):.4f} -> C3 {auc('C3 (+IDN)', sp):.4f} "
                 f"(new-writer {auc('C1 (fixed building)', sp, 'AUC_new_writer_view'):.4f} -> {auc('C3 (+IDN)', sp, 'AUC_new_writer_view'):.4f})" for sp in splits))
record("C0 vs C1 (3.0 vs fixed building)", all(auc("C1 (fixed building)", sp) >= auc("C0 (3.0 building)", sp) - 0.002 for sp in splits),
       "; ".join(f"{sp}: C0 {auc('C0 (3.0 building)', sp):.4f} vs C1 {auc('C1 (fixed building)', sp):.4f}" for sp in splits))
record("C4 passing tests add value", all(auc("C4 (+passing tests)", sp) > auc("C3 (+IDN)", sp) for sp in splits),
       "; ".join(f"{sp}: C3 {auc('C3 (+IDN)', sp):.4f} vs C4 {auc('C4 (+passing tests)', sp):.4f}" for sp in splits))
record("Within-specialty check (C3 vs C1)", builtins.sum(auc("C3 (+IDN)", sp, f"AUC_{s}") > auc("C1 (fixed building)", sp, f"AUC_{s}") for sp in splits for s in SPECIALTIES) >= 6,
       ", ".join(f"{s}: {auc('C1 (fixed building)', 'test c25-c26', f'AUC_{s}'):.3f}->{auc('C3 (+IDN)', 'test c25-c26', f'AUC_{s}'):.3f}" for s in SPECIALTIES) + " (test split)")

In [0]:
OUT_COLS = KEYS + FEATURE_COLS + ["PULL_DT"]
out = fa.select(*OUT_COLS)
out.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(OUTPUT_PATH)
run1 = spark.read.format("delta").load(OUTPUT_PATH)

# Second run: rebuild the features for the latest two cohorts from scratch (same code path) and compare schema and values
run2 = (feat_all.filter(F.col("cohort_id").isin(cohorts.cohort_id.iloc[-2:].tolist())).select(*OUT_COLS))
same_schema = [(f.name, f.dataType.simpleString()) for f in run1.schema] == [(f.name, f.dataType.simpleString()) for f in run2.schema]
diff = (run1.filter(F.col("cohort_id").isin(cohorts.cohort_id.iloc[-2:].tolist())).drop("PULL_DT")
        .exceptAll(run2.drop("PULL_DT")).count())
record("S6 fixed schema on two runs", same_schema and diff == 0 and run1.columns == OUT_COLS,
       f"{len(OUT_COLS)} columns ({len(FEATURE_COLS)} features), schema identical: {same_schema}, differing rows on rerun of last 2 cohorts: {diff}")
n_out = run1.count()
record("Output written", n_out == n_hcp * 28, f"{n_out:,} rows -> {OUTPUT_PATH}")

In [0]:
def ok(*ids):
    return all(CHECKS[i]["passed"] for i in ids)
gates = [
    ("G0", "Spine reproduces 33_2 coverage", ok("F1/S1 spine coverage", "F1 one practice per HCP")),
    ("G0", "Metric-view writer helper in place (= KER-10)", ok("F2 writer helper = KER-10")),
    ("G0", "Snapshot change rate measured (F4)", ok("F4 change rate measured")),
    ("G0", "Building fix specified and applied (F6)", ok("F6/S3 building fix")),
    ("G0", "Data requests DR-01/DR-02 sent", False),
    ("G1", "Build block for all 28 training/test/holdout cohorts", ok("Build: one row per HCP x cohort", "Output written")),
    ("G1", "Leakage and self-exclusion tests pass", ok("S2 toy self-exclusion test", "S2 self-exclusion on real data", "S2 leakage test", "F5 anchor spine / S2 time rule", "P3 independent recompute (practice, latest cohort)")),
    ("G1", "Signal reproduced on metric-view writers", ok("P4 signal restated on metric-view writers")),
    ("G1", "Fixed schema on two runs", ok("S6 fixed schema on two runs")),
    ("G1", "C1-C3 run and C2/C3 beat C1 (proxy model)", ok("S4 practice + IDN add value over fixed building")),
]
goals = [
    ("S1", "Spine for every HCP-month, placeholders removed, 33_2 counts reproduced", ok("F1/S1 spine coverage", "Build: one row per HCP x cohort")),
    ("S2", "Peer writing only from before the anchor; HCP never in own group", ok("S2 toy self-exclusion test", "S2 self-exclusion on real data", "F5 anchor spine / S2 time rule")),
    ("S3", "3.0 building defects fixed", ok("F6/S3 building fix", "P3 missing-state rules")),
    ("S4", "Practice and IDN add value over fixed building", ok("S4 practice + IDN add value over fixed building")),
    ("S5", "Writers from KER-10 active_nbrx_writers", ok("F2 writer helper = KER-10")),
    ("S6", "Fixed schema; feature dictionary written", ok("S6 fixed schema on two runs")),
]
gate_df = pd.DataFrame(gates, columns=["gate", "criterion", "passed"])
goal_df = pd.DataFrame(goals, columns=["goal", "criterion", "achieved"])
check_df = pd.DataFrame([{"check": k, **v} for k, v in CHECKS.items()])
display(check_df); display(gate_df); display(goal_df)

In [0]:
final = spark.read.format("delta").load(OUTPUT_PATH)
print(f"{final.count():,} rows x {len(final.columns)} columns | cohorts {final.select('cohort_id').distinct().count()} | HCPs {final.select('PROVIDER_ID').distinct().count():,}")
display(final.filter(F.col("cohort_id") == last_c).orderBy(F.desc("COLOC_PRAC_PEER_WRITERS_3M")).limit(100))

In [0]:
fr = final.filter(F.col("cohort_id") == last_c).select(
    *[F.avg(F.col(c).isNotNull().cast("double")).alias(f"{c}__filled") for c in FEATURE_COLS],
    *[F.avg((F.col(c) != 0).cast("double")).alias(f"{c}__nonzero") for c in FEATURE_COLS]).toPandas().T[0]
fill = pd.DataFrame({"feature": FEATURE_COLS, "status": ["Build"] * 7 + ["Test"] * 10,
                     "filled_%": [round(100 * fr[f"{c}__filled"], 2) for c in FEATURE_COLS],
                     "non_zero_of_filled_%": [round(100 * fr[f"{c}__nonzero"], 2) for c in FEATURE_COLS]})
print(f"Fill rates for cohort {last_c} ({n_hcp:,} HCPs)")
dbutils.fs.rm(STAGE_PATH, True)   # staging copy no longer needed (final output is at OUTPUT_PATH)
display(fill)

## Feature dictionary

Grain: one row per `PROVIDER_ID` x `cohort_id` (28 claims cohorts, 294,727 HCPs, 8,252,356 rows). Keys: `PROVIDER_ID`, `cohort_id`, `SVC_DT_M` (= cohort `lookback_end_date`), `HCP_COHORT_ID`; plus `PULL_DT`. "Writer" = counted by KER-10 `active_nbrx_writers` (`mv_kerendia_nbrx`) in any week of the 3 months ending with `SVC_DT_M`. Peers are other universe HCPs (different BH_ID) in the same group; the HCP is never counted. Fill rates are for c28.

| Feature | Status | Level | Definition | Missing / special values | Filled % | Non-zero % of filled | Recommendation |
| --- | --- | --- | --- | --- | --- | --- | --- |
| `COLOC_BLDG_PEER_WRITERS_3M` | Build | Building (lat, long, BUILDING) | Number of peer writers in the same building | null = no geocoded building; 0 = building but no peer writer | 95.40 | 19.33 | Keep (replaces 3.0 `NUM_HCPS_IN_BUILDING`) |
| `COLOC_BLDG_PEER_WRITER_FLAG` | Build | Building | 1 if `COLOC_BLDG_PEER_WRITERS_3M` > 0 else 0 | null = no building | 95.40 | 19.33 | Keep (replaces 3.0 `BUILDING_COLOCATION_FLAG`) |
| `COLOC_PRAC_PEER_WRITERS_3M` | Build | Practice (L1 = BEST_AFFL rank-1 BHO_ID) | Number of peer writers in the same practice | null = no rank-1 practice; 0 = alone or no peer writer | 95.98 | 29.52 | Keep |
| `COLOC_PRAC_PEER_SHARE_3M` | Build | Practice | Peer writers / (practice universe HCPs - 1) | null = no practice or alone in practice | 79.09 | 35.82 | Keep |
| `COLOC_PRAC_HCP_COUNT` | Build | Practice | Universe HCPs (BH_IDs) in the practice, incl. the HCP | null = no practice | 95.98 | 100 | Keep |
| `COLOC_IDN_PEER_SHARE_3M` | Build | IDN (hierarchy L4, placeholders removed) | Peer writers / (IDN universe HCPs - 1) | null = no real IDN or alone in IDN | 63.42 | 86.26 | Keep |
| `COLOC_IDN_PEER_SHARE_ZERO_FLAG` | Build | IDN | 1 if `COLOC_IDN_PEER_SHARE_3M` = 0, 0 if > 0 | null when the share is null | 63.42 | 13.74 | Keep |
| `COLOC_HQ_PEER_SHARE_3M` | Test (T1) | HQ (hierarchy L3, placeholders removed) | Peer writers / (L3 universe HCPs - 1) | null = no L3 or alone | 83.01 | 55.03 | Promote: lift within every L1 band; in C4 |
| `COLOC_PRAC_PEER_WRITERS_NEPH_3M` | Test (T2) | Practice | Peer writers whose NBRx `specialty_group` = NEPH | as practice count | 95.98 | 13.48 | Promote (in C4) |
| `COLOC_PRAC_PEER_WRITERS_CARD_3M` | Test (T2) | Practice | Same, CARD | as practice count | 95.98 | 12.78 | Promote (in C4) |
| `COLOC_PRAC_PEER_WRITERS_ENDO_3M` | Test (T2) | Practice | Same, ENDO | as practice count | 95.98 | 4.51 | Promote (in C4) |
| `COLOC_PRAC_PEER_WRITERS_PCP_3M` | Test (T2) | Practice | Same, PCP | as practice count | 95.98 | 18.77 | Promote (in C4) |
| `COLOC_PRAC_HF_PEER_WRITERS_3M` | Test (T2) | Practice | Peer writers with an HF indication NBRx (`indication_group` contains 'HF') | as practice count | 95.98 | 21.43 | Drop: no lift over any peer writer (16.9% vs 16.6%) |
| `COLOC_HCP_IN_IDN` | Test (T3) | IDN | 1 if the practice belongs to a real IDN, 0 if not | null = no practice | 95.98 | 74.39 | Drop: no lift (8.6% vs 9.4%) |
| `COLOC_IDN_HF_CKD_PATIENTS` | Test (T4) | PACE `ph_hs_metrics_static` at L4, else L3 | `HF_CKD_Patients_top` of the HCP's IDN / HQ | null = no IDN/HQ or not in PACE | 91.94 | 99.39 | Drop for now: negative lift; definitions unconfirmed (DR-08) |
| `COLOC_IDN_H1_HOSPITALS` | Test (T4) | PACE, as above | `H1_COUNT` | as above | 60.46 | 100 | Drop for now (same reason) |
| `COLOC_IDN_FORMULARY_WIN` | Test (T4) | PACE, as above | 1 if `formulary_win_date` < anchor month, else 0 | null = no IDN/HQ | 94.59 | 10.97 | Drop for now (same reason) |

Other outputs: IDN adoption view (B1) at `.../notebook_output/reference/idn_adoption_view`; affiliation snapshot archive (O1) at `.../notebook_output/reference/affiliation_snapshots/snapshot_dt=YYYY-MM-DD`.

# NBA 3.1 - 33 Colocation and IDN features

**Plan:** `33_3_colocation_idn_nba_3_1_readiness_plan.md` | **Run:** 2026-11-19 | **Universe:** `HCP_Universe_without_call_hcps` (294,727) | **Cohorts:** 28 claims cohorts c1-c28 | **Writers:** KER-10 `active_nbrx_writers` in `kerendia_metric_views.mv_kerendia_nbrx` | **Output:** `kerendia_nba_3_1_volume/.../notebook_output/hcp_features/colocation_idn_features` - 8,252,356 rows, 22 columns (7 Build + 10 Test features)

### Checks that passed (24 of 24 recorded checks)
* **F1 / S1 spine:** L1 282,881 HCPs in 90,861 practices, real IDN 210,425 HCPs in 33,062 IDNs - all equal to `33_2`. One rank-1 practice per HCP; one hierarchy row per practice.
* **F2 writer helper:** equals `MEASURE(active_nbrx_writers)` in all 34 months (2024-01 to 2026-10), 0 mismatches.
* **F3 new-writer view**, **F4** change rate (practice 6.57% / 12.2%, IDN 3.27% / 6.31% per quarter, call-plan HCPs only), **F5** time rule (0 writer weeks on/after the anchor month).
* **Self-exclusion:** toy test with a known answer, and on real data (0 HCPs alone with a non-zero count, 0 counts above peers, all shares in [0,1]); independent recompute of the practice count for c28 with a window function: 0 mismatches over 282,845 HCPs.
* **Missing-state rules** (null / 0 / share null when alone): 0 violations.
* **F6 / S3 building fix:** filled for all 281,159 geocoded HCPs x 28 (vs 1,355,844 rows under 3.0 logic); 303,315 rows where 3.0 counted the HCP itself are corrected; 6,516,608 former nulls are now 0.
* **P4 signal** on metric-view writers (c17, 12-month outcome): every `33_2` finding holds in direction; rates are about 1-2 points higher than on LRx V7 (baseline 10.6% vs 9.7%), e.g. Building Yes / Practice Yes 24.6% vs 23.1%. IDN share rho 0.097 (07_2: 0.125).
* **T1-T4** run; **B1** IDN view written; **O1** snapshot archived; **O2** no cohort-to-cohort jump > 5 points.
* **Experiments (proxy model, N14), AUC test c25-c26 / holdout c27-c28:** base 0.824 / 0.832; C0 3.0 building 0.843 / 0.849; C1 fixed building 0.844 / 0.850; C2 + practice 0.857 / 0.863; C3 + IDN 0.861 / 0.865; C4 + HQ and specialty peers 0.863 / 0.868. New-writer view C1 -> C3: 0.715 -> 0.749. C3 beats C1 within NEPH, CARD, ENDO and PCP.
* **S6:** schema identical and 0 differing rows when the last 2 cohorts are rebuilt.

### Goals (Section 2)
| Goal | Achieved |
| --- | --- |
| S1 Spine for every HCP-month, placeholders removed, 33_2 counts reproduced | Yes (building 281,159 vs 282,465 in 33_2: 33_2 also counted 1,306 HCPs with a geocode but no building name) |
| S2 Peer writing only before the anchor; HCP never in own group | Yes |
| S3 3.0 building defects fixed | Yes |
| S4 Practice and IDN add value over the fixed building | Yes on the proxy model; must be confirmed in the real NBA 3.1 model |
| S5 Writers from KER-10 | Yes |
| S6 Fixed schema and feature dictionary | Yes |

**Gates:** G0 passes except "data requests DR-01 / DR-02 sent" (still Open, can only be done by the team). G1 passes on all criteria, with C1-C3 run on the proxy model.

### What was not possible
* **Affiliation and hierarchy history (D7, DR-01/02):** only the Q4-2026 snapshot exists, so past cohorts use today's practice and IDN. About 6.6-12.2% of call-plan HCPs change practice per quarter. This can inflate the practice and IDN signal in early cohorts.
* **Real NBA 3.1 model runs (C0-C5):** a gradient-boosting proxy on own Kerendia history plus this block was used instead. SHAP share of the block and C5 (combined with other 3.1 families) were not run.
* **Sending DR-01 / DR-02:** can only be done by the team. The F4 change rate is logged in `01_data_improvement_requests.md`.
* **F4 for the full universe:** the quarterly copies only hold call-plan HCPs (68,945 / 83,985 matched).
* **T4 PACE fields:** definitions are not confirmed (Q4, DR-08), and the univariate lift is negative, so these stay as Test only.
* **HF peers, IDN membership, all-HCP peers:** tested, no lift over the Build features, recommended to drop. All-HCP peers are not written to the output.
* **B2 reason codes:** only specified (rule cell below). They need the model's driver output to apply.